In [1]:
from ase.io import read
from ase.build import make_supercell
from matplotlib import rc

rc("text", usetex=True)
import matplotlib.pylab as pylab

params = {"legend.fontsize": "14", "axes.labelsize": 20, "axes.titlesize": 20, "xtick.labelsize": 16, "ytick.labelsize": 16}
pylab.rcParams.update(params)
import numpy as np
import glob

try:
    from tqdm import tqdm as _tqdm

    _TQDM_AVAILABLE = True
except ImportError:
    _TQDM_AVAILABLE = False


def _progress(iterable, **kwargs):
    """Wrap an iterable with tqdm if available, else return as-is."""
    if _TQDM_AVAILABLE:
        return _tqdm(iterable, **kwargs)
    return iterable

In [ ]:
# load the molecular dynamics trajectory
traj_directory = "/Users/jackyang-macmini/Dropbox/Documents/projects/artificial_intelligence/LiPSCl-diffusion/nve_md/FT_MACE_long_traj/200_K/"
all_traj_files = glob.glob(traj_directory + "nve_md_*.traj")
traj = []
for i in range(len(all_traj_files)):
    print(f"Loading trajectory file {i+1}/{len(all_traj_files)}: {all_traj_files[i]}")
    traj_name = traj_directory + f"nve_md_{i+1}.traj"
    traj += read(traj_name, index=":")
print("loaded", len(traj), "frames")

# loop through the trajectory and update each frame's positions to be the wrapped positions, and compute the average structure
averaged_structure = traj[0].copy()
averaged_structure.positions[:] = 0.0

coms_all_frames = []
all_masses = np.array([atom.mass for atom in traj[0]], dtype=float)
total_mass = all_masses.sum()

for frame in _progress(traj, desc="Computing averaged structure", leave=False):
    # Vectorized wrapping (fractional -> wrapped Cartesian) handled by ASE.
    wrapped_positions = frame.get_positions(wrap=True)
    frame.positions[:] = wrapped_positions

    averaged_structure.positions += wrapped_positions
    coms_all_frames.append(np.average(wrapped_positions, axis=0, weights=all_masses))

averaged_structure.positions /= len(traj)
print("computed averaged structure, and all centers of masses")

# now loop through the trajectory, and for each frame, calculate the displacement of each atom from its position in the averaged structure, taking into account the periodic boundary conditions, and normalise the displacement by the square root of the mass of each atom.
atomic_displacements = []

averaged_positions = averaged_structure.positions

for frame_index, frame in _progress(enumerate(traj), total=len(traj), desc="Compute mass-normalized displacements", leave=False):
    drift = coms_all_frames[frame_index] - center_of_mass_averaged_structure

    # Vectorized displacement for all atoms in this frame.
    delta_r = frame.positions - drift - averaged_positions

    # Apply periodic boundary conditions with minimum-image convention.
    cell_diag = frame.cell.diagonal()
    delta_r -= np.round(delta_r / cell_diag) * cell_diag

    # Mass-normalize and flatten to shape (3N,).
    atomic_displacements.append(delta_r)

element = "Li"
input_atom_order = read(traj_directory + "CONTCAR").get_chemical_symbols()
framework_indices = [i for i, symbol in enumerate(input_atom_order) if symbol != element]
framework_indices = list(np.array([[3 * i, 3 * i + 1, 3 * i + 2] for i in framework_indices]).reshape(-1))
framework_displacements = np.array([displacement[framework_indices] for displacement in atomic_displacements])
print("computed displacements for the framework atoms")